# Agent Wolf (Phase 4: MQTT Subscriber + Publisher)

This notebook subscribes to sheep/tick topics and publishes wolf state and wolf-driven events.

Phase 4 scope:
- Subscribe: `.../sim/sheep/state`, `.../sim/tick`
- Publish: `.../sim/wolf/state`, `.../sim/events`

This notebook does not implement dashboard or controller behavior.

In [ ]:
# Cell purpose: import dependencies and connect to MQTT using project config.
from datetime import datetime, timezone
import json
import random
import time

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher

config = load_config()
print(f"Loaded config. Primary MQTT profile: {config.mqtt.host}:{config.mqtt.port}")

connector = MqttConnector(config.mqtt, client_id_suffix="agent-wolf-phase4")
connector.connect()
connected = connector.wait_for_connection(timeout=5)
print(f"MQTT connected: {connected}")

publisher = MqttPublisher(connector)
print("STARTUP_OK: agent_wolf connected and ready")

In [ ]:
# Cell purpose: define topic names and phase-4 wolf state variables.
base_topic = config.mqtt.base_topic.rstrip("/")
tick_topic = f"{base_topic}/sim/tick"
sheep_state_topic = f"{base_topic}/sim/sheep/state"
wolf_state_topic = f"{base_topic}/sim/wolf/state"
events_topic = f"{base_topic}/sim/events"

simulation_cfg = config.simulation
wolf_count = simulation_cfg.initial_wolves if simulation_cfg is not None else 8
wolf_capacity = simulation_cfg.wolf_predation_capacity if simulation_cfg is not None else 1
grid_width = simulation_cfg.grid_width if simulation_cfg is not None else 10
grid_height = simulation_cfg.grid_height if simulation_cfg is not None else 10
total_cells = max(1, int(grid_width) * int(grid_height))
seed = (simulation_cfg.seed + 1000) if (simulation_cfg is not None and simulation_cfg.seed is not None) else 1042
rng = random.Random(seed)

last_tick = 0
last_sheep_count = simulation_cfg.initial_sheep if simulation_cfg is not None else 30

print("Subscribe topics:")
print(f"- {sheep_state_topic}")
print(f"- {tick_topic}")
print("Publish topics:")
print(f"- {wolf_state_topic}")
print(f"- {events_topic}")

In [ ]:
# Cell purpose: register callbacks to consume sheep/tick messages and publish wolf messages.
def on_message(client, userdata, msg):
    global last_sheep_count, last_tick, wolf_count

    try:
        payload = json.loads(msg.payload.decode("utf-8"))
    except Exception:
        print(f"Skipping non-JSON payload on topic {msg.topic}")
        return

    if msg.topic == sheep_state_topic:
        last_sheep_count = int(payload.get("sheep_count", last_sheep_count))
        return

    if msg.topic == tick_topic:
        tick = int(payload.get("tick", last_tick + 1))
        last_tick = tick

        sheep_density = max(0.0, min(1.0, float(last_sheep_count) / float(total_cells)))
        potential_predation = int(round(wolf_count * wolf_capacity * sheep_density))
        if potential_predation == 0 and wolf_count > 0 and last_sheep_count > 0 and rng.random() < sheep_density:
            potential_predation = 1
        predation_count = min(potential_predation, max(0, last_sheep_count))

        births = 1 if (rng.random() < 0.04 and wolf_count > 0) else 0
        deaths = 1 if (rng.random() < 0.02 and wolf_count > 0) else 0
        wolf_count = max(0, wolf_count + births - deaths)

        wolf_state_payload = {
            "tick": tick,
            "seed": seed,
            "wolf_count": wolf_count,
            "predation_capacity": wolf_capacity,
            "estimated_predation": predation_count,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        }

        events_payload = {
            "tick": tick,
            "seed": seed,
            "source": "wolf",
            "wolf_births": births,
            "wolf_deaths": deaths,
            "estimated_predation": predation_count,
        }

        publisher.publish_json(wolf_state_topic, json.dumps(wolf_state_payload), qos=0)
        publisher.publish_json(events_topic, json.dumps(events_payload), qos=0)

        print(
            f"tick={tick} sheep={last_sheep_count} wolves={wolf_count} "
            f"predation={predation_count} births={births} deaths={deaths} | published"
        )

connector.client.on_message = on_message
connector.client.subscribe(sheep_state_topic, qos=0)
connector.client.subscribe(tick_topic, qos=0)
print("Subscriptions active.")

In [ ]:
# Cell purpose: keep this notebook alive to process incoming MQTT messages.
print("Wolf agent loop running. Interrupt the cell to stop.")
try:
    while True:
        time.sleep(0.2)
except KeyboardInterrupt:
    print("Stopping wolf agent loop.")
finally:
    connector.disconnect()
    print("MQTT disconnected.")